In [1]:
import pandas as pd

running = pd.read_csv("../data/processed/bigwest_2021_2026_running.csv")
field = pd.read_csv("../data/processed/bigwest_2021_2026_field.csv")

athletes = pd.concat([
    running[["athlete_id", "athlete_name"]],
    field[["athlete_id", "athlete_name"]]
]).drop_duplicates(subset=["athlete_id"])

In [2]:
athlete_ids = (
    athletes["athlete_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda x: x.ne("")]
    .unique()
)

In [3]:
url = "https://api.parse.bot/scraper/1b66c7cb-769a-4301-82ff-8d308afee7ad/get_athlete_results"

headers = {
    "X-API-Key": "pmx_127bea8970ed7f3b4ddaec353e4310e8",
    "API-Snapshot-Version": "5"
}

In [ ]:
import requests
import pandas as pd
import time

# Load existing results
results_df = pd.read_csv("big-west-predictor/data/raw/bigwest_regular_season_partial.csv")

# Track completed athletes
completed_ids = set(
    results_df["athlete_id"].dropna().astype(str).str.strip()
)

# Find athletes still needing data
remaining_ids = [
    str(x).strip()
    for x in athlete_ids
    if str(x).strip() not in completed_ids
]

remaining_ids = list(dict.fromkeys(remaining_ids))

print(f"Starting collection for {len(remaining_ids)} athletes")

new_rows = []

for i, athlete_id in enumerate(remaining_ids):

    try:
        response = requests.get(
            url,
            headers=headers,
            params={"athlete_id": athlete_id},
            timeout=120
        )

        # Stop when rate-limited
        if response.status_code == 429:
            print("Rate limit reached.")
            print("Retry after:", response.headers.get("Retry-After"))
            print("Daily remaining:", response.headers.get(
                "x-ratelimit-daily-remaining"
            ))
            break

        response.raise_for_status()

        result = response.json()

        if result.get("status") != "success":
            print(f"Unsuccessful response for {athlete_id}")
            continue

        athlete = result["data"]

        athlete_rows = []

        for performance in athlete.get("performances", []):

            meet_date = str(performance.get("meet_date", ""))

            # Remove incorrectly parsed records
            if not any(month in meet_date for month in [
                "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
            ]):
                continue

            athlete_rows.append({
                "athlete_id": athlete_id,
                "athlete_name": athlete.get("name"),
                "team": athlete.get("team"),
                **performance
            })

        if athlete_rows:
            new_rows.extend(athlete_rows)

            # Combine old and new results
            updated_df = pd.concat(
                [results_df, pd.DataFrame(new_rows)],
                ignore_index=True
            )

            updated_df = updated_df.drop_duplicates()

            # Save after each athlete
            updated_df.to_csv(
                "../data/raw/bigwest_regular_season_partial.csv",
                index=False
            )

        completed_ids.add(athlete_id)

        print(
            f"Completed {i + 1}/{len(remaining_ids)}: "
            f"{athlete.get('name')}"
        )

        # Stop near daily quota
        daily_remaining = response.headers.get(
            "x-ratelimit-daily-remaining"
        )

        if daily_remaining is not None and int(daily_remaining) <= 1:
            print("Daily API quota almost exhausted.")
            break

    except (requests.RequestException, ValueError, KeyError, TypeError) as e:
        print(f"Failed {athlete_id}: {e}")

    # Stay within the 5 requests/minute refill rate
    time.sleep(13)

print("Collection stopped. Progress saved.")

In [5]:
results_df = pd.read_csv("../data/raw/bigwest_regular_season_partial.csv")

completed_ids = set(
    results_df["athlete_id"].dropna().astype(str).str.strip()
)

remaining_ids = set(
    str(x).strip() for x in athlete_ids
) - completed_ids

print("Total athletes:", len(athlete_ids))
print("Athletes collected:", len(completed_ids))
print("Athletes remaining:", len(remaining_ids))
print("Total performance records:", len(results_df))

Total athletes: 1514
Athletes collected: 145
Athletes remaining: 1369
Total performance records: 7633


In [7]:
running['school'].unique()

<StringArray>
[         'UC Davis',  'UC Santa Barbara', 'Cal St. Fullerton',
          'Cal Poly',         'UC Irvine',    'Long Beach St.',
      'UC Riverside',              'CSUN',   'CSU Bakersfield',
            'Hawaii',      'UC San Diego']
Length: 11, dtype: str